In [149]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score

# Data Loading and Initial Exploration

In [150]:
df = pd.read_csv("titanic_data_updated.csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
406,407,no,third,"Widegren, Mr. Carl/Charles Peter",male,51.0,0,0,347064,7.7500,NaN,S
285,286,no,third,"Stankovic, Mr. Ivan",male,33.0,0,0,349239,8.6625,NaN,C
837,838,no,third,"Sirota, Mr. Maurice",male,NaN,0,0,392092,8.0500,NaN,S
704,705,no,third,"Hansen, Mr. Henrik Juul",male,26.0,1,0,350025,7.8542,NaN,S
286,287,yes,third,"de Mulder, Mr. Theodore",male,30.0,0,0,345774,9.5000,NaN,S


In [151]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


# Feature Engineering

In [152]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
464,465,no,third,"Maisner, Mr. Simon",male,NaN,0,0,A/S 2816,8.0500,Missing,S,1,M
486,487,yes,first,"Hoyt, Mrs. Frederick Maxfield (Jane Anne Forby)",female,35.0,1,0,19943,90.0000,C93,S,2,C
181,182,no,second,"Pernot, Mr. Rene",male,NaN,0,0,SC/PARIS 2131,15.0500,Missing,C,1,M
73,74,no,third,"Chronopoulos, Mr. Apostolos",male,26.0,1,0,2680,14.4542,Missing,C,2,M
582,583,no,second,"Downton, Mr. William James",male,54.0,0,0,28403,26.0000,Missing,S,1,M


In [153]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [154]:
X = df.drop('Survived', axis=1)
y = df['Survived']

# Randrom state , Stratify and Train Test Split

In [155]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Implementation of Preprocessor Pipeline

In [156]:
# Pipeline
# Numerical Values
p1 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ]
)

In [157]:
categories = [['third', 'second', 'first']]

In [158]:
# Pipeline
# Categorical Column
p3 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=categories)),
        ('scaler', MinMaxScaler())
    ]
)

In [159]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipe_1', p1, ['Age']),
        ('pipe_2', p2, ['Fare', 'Family_Size']),
        ('pipe_3', p3, ['Embarked', 'Sex', 'Deck']),
        ('pipe_4', p4, ['Pclass']),
    ],
    remainder='drop'
)
preprocessor

,transformers,"[('pipe_1', ...), ('pipe_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


# Label Encoding

In [160]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

# Training the Model

In [161]:
lr_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
    ]
)
lr_model

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipe_1', ...), ('pipe_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [162]:
lr_model.fit(X_train, y_train)

lr_model['model'].classes_
lr_model['model'].coef_
lr_model['model'].intercept_

array([1.86583369])

In [163]:
y_pred = lr_model.predict(X_test)

lr_model.predict_proba(X_test)

array([[0.86250724, 0.13749276],
       [0.92185031, 0.07814969],
       [0.74629618, 0.25370382],
       [0.92223345, 0.07776655],
       [0.23460179, 0.76539821],
       [0.41618011, 0.58381989],
       [0.23706975, 0.76293025],
       [0.58977302, 0.41022698],
       [0.52751483, 0.47248517],
       [0.79720633, 0.20279367],
       [0.77216932, 0.22783068],
       [0.82447686, 0.17552314],
       [0.31526778, 0.68473222],
       [0.70897134, 0.29102866],
       [0.41869444, 0.58130556],
       [0.73947197, 0.26052803],
       [0.44218511, 0.55781489],
       [0.8626754 , 0.1373246 ],
       [0.79673722, 0.20326278],
       [0.22398311, 0.77601689],
       [0.8626754 , 0.1373246 ],
       [0.2038076 , 0.7961924 ],
       [0.86351383, 0.13648617],
       [0.43438672, 0.56561328],
       [0.859127  , 0.140873  ],
       [0.03125334, 0.96874666],
       [0.48026201, 0.51973799],
       [0.61935982, 0.38064018],
       [0.79192773, 0.20807227],
       [0.79577908, 0.20422092],
       [0.

# Evaluation

In [164]:
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)
precision = precision_score(y_test, y_pred)
print(precision)
recall = recall_score(y_test, y_pred)
print(recall)

0.7597765363128491
0.6666666666666666
0.7536231884057971
